| **Retriever Type**                   | **Purpose**                                                                         | **Ideal Usage Scenario**                                                                                                       |
| ------------------------------------ | ----------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------ |
| **VectorStoreRetriever**             | Retrieve documents based on vector similarity search (e.g. FAISS, Chroma, Pinecone) | Basic RAG (Retrieval-Augmented Generation) setup for fetching semantically relevant documents                                  |
| **ContextualCompressionRetriever**   | Compresses retrieved docs using an LLM to return only the most relevant parts       | When using large context docs (PDFs, transcripts) where only a portion of each doc is relevant                                 |
| **MultiQueryRetriever**              | Generates multiple rephrased queries to improve retrieval coverage                  | When a user query can be ambiguous or phrased differently; improves diversity of retrieved documents                           |
| **ParentDocumentRetriever**          | Retrieves a full parent document instead of the split chunk                         | When a chunk is matched but full document context is required (e.g., blog, article, contract analysis)                         |
| **BM25Retriever**                    | Keyword-based retriever using classic BM25 algorithm                                | When you want to retrieve based on exact keyword matches (useful for legal, medical, or short documents with precise language) |
| **EnsembleRetriever**                | Combines multiple retrievers (e.g., vector + BM25) with weighted scoring            | When you want to merge semantic and keyword-based search for better hybrid performance                                         |
| **TimeWeightedVectorStoreRetriever** | VectorStoreRetriever + time-based decay to prefer recent documents                  | Best for chatbot memory or news search where **recency** of information is important                                           |
| **TavilySearchAPIRetriever**         | Uses external web search engine via Tavily API to retrieve **live web data**        | When querying real-world events, breaking news, or updated information outside your local docs or vector database              |


In [6]:
#1)VectorStoreRetriever	Vector-based	Embedding similarity search	General-purpose RAG
# ConversationalRetrievalChain


#from langchain.chains import ConversationalRetrievalChain
from langchain_core.tools import StructuredTool

# 1️⃣ Imports
from langchain_core.tools import Tool
from langchain_classic.agents import initialize_agent, AgentType
from langchain_community.vectorstores import FAISS
#from langchain.embeddings import OpenAIEmbeddings 
from langchain_classic.chains import RetrievalQA
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.memory import VectorStoreRetrieverMemory
from langchain_classic.chains import ConversationalRetrievalChain

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
#from langchain.chains import LLMChain
from langchain_classic.memory import ConversationBufferMemory


# 1. Setup

import os
from dotenv import load_dotenv


# 2️⃣ Load API keys
load_dotenv(".env")
google_api_key=os.getenv("GOOGLE_API_KEY")

# 3️⃣ Setup LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite", temperature=0)

# 4️⃣ Create Vector DB (Retriever)

with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

# 🧠 Split the text into smaller chunks

from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts = splitter.split_text(text_data)
embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001")
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

# 3. Conversational RAG chain
rag_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=retriever,
    return_source_documents=False
)

# 4. Memory for chat history
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# 5. Wrap RAG as a StructuredTool
def rag_tool_fn(question: str) -> str:
    return rag_chain.invoke({
        "question": question,
        "chat_history": []
    })["answer"]

rag_tool = StructuredTool.from_function(
    name="RAG_QA",
    description="Use this to answer questions about LangChain.",
    func=rag_tool_fn
)

# 6. Create agent
agent = initialize_agent(
    tools=[rag_tool],
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    memory=memory,
    handle_parsing_errors=True
)

# 7. Run conversation
print("1️⃣ First question")
res1 = agent.run("What is LangChain?")
print("Answer:", res1)

print("\n2️⃣ Follow-up")
res2 = agent.run("Who created it?")
print("Answer:", res2)

print("\n3️⃣ Ask again")
res3 = agent.run("Explain LangChain again simply.")
print("Answer:", res3)


1️⃣ First question


> Entering new AgentExecutor chain...


C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Thought: Do I need to use a tool? Yes
Action: RAG_QA
Action Input: What is LangChain?

C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



Observation: Based on the provided context, LangChain is a framework for building applications with LLMs (Large Language Models). It was created by Harrison Chase and supports features such as RAG, agents, memory, and tools, making it commonly used in chatbots, document Q&A, and AI workflows.
Thought:

C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


... (5 outputs left out)

C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Do I need to use a tool? No
AI: LangChain was created by Harrison Chase.

> Finished chain.
Answer: LangChain was created by Harrison Chase.

3️⃣ Ask again


> Entering new AgentExecutor chain...


C:\Users\user\anaconda3\envs\W18_Langchain\Lib\site-packages\langchain_google_genai\chat_models.py:3919: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Thought: Do I need to use a tool? No
AI: Think of LangChain as a helpful toolkit or bridge that makes it easier for developers to build applications using Large Language Models (like ChatGPT). Instead of writing everything from scratch, LangChain gives you pre-made building blocks to connect AI models to other things, such as documents to search through, memory to remember past conversations, and tools to perform specific tasks.

> Finished chain.
Answer: Think of LangChain as a helpful toolkit or bridge that makes it easier for developers to build applications using Large Language Models (like ChatGPT). Instead of writing everything from scratch, LangChain gives you pre-made building blocks to connect AI models to other things, such as documents to search through, memory to remember past conversations, and tools to perform specific tasks.
